In [1]:
# !pip install optimum[gptq]
!pip install auto_gptq
!pip install optimum
!pip install transformers
!pip install huggingface_hub
!pip install -U accelerate bitsandbytes datasets peft transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   

In [2]:
!pip install sentence-transformers


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import csv

nltk.download('punkt')

# Initialize models and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/CodeQwen1.5-7B-Chat",
    torch_dtype="auto",
    device_map="auto"
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("Qwen/CodeQwen1.5-7B-Chat")

# Initialize semantic similarity model
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_semantic_similarity(code1, code2):
    # Get embeddings for both code snippets
    embedding1 = semantic_model.encode(code1)
    embedding2 = semantic_model.encode(code2)

    # Calculate cosine similarity
    similarity = np.dot(embedding1, embedding2) / (np.linalg.norm(embedding1) * np.linalg.norm(embedding2))
    return similarity

def get_functional_similarity(code1, code2):
    prompt = f"""Compare the following two code snippets and rate their functional/conceptual similarity on a scale of 0 to 1, where 1 means identical functionality and 0 means completely different functionality. Only output the numerical score.

    Code 1:
    {code1}

    Code 2:
    {code2}"""

    messages = [
        {"role": "system", "content": "You are a code analysis assistant that evaluates functional similarity between code snippets."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=10,
            temperature=0.1
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    try:
        similarity_score = float(response.strip())
        return max(0.0, min(1.0, similarity_score))  # Ensure score is between 0 and 1
    except:
        return 0.0

def translate_verilog_to_c(verilog_code):
    prompt = f"""Translate the following Verilog code to C code.
    Only output the C code, without any explanations or additional text.

    Verilog code:
    {verilog_code}"""

    messages = [
        {"role": "system", "content": "You are a code translation assistant that converts Verilog to C code."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response.strip()

def load_data(src_file, tgt_file):
    with open(src_file, 'r') as sf, open(tgt_file, 'r') as tf:
        src_lines = sf.readlines()
        tgt_lines = tf.readlines()
    assert len(src_lines) == len(tgt_lines), "Mismatch in number of source and target lines"
    return [(src.strip(), tgt.strip()) for src, tgt in zip(src_lines, tgt_lines)]

# Create output directories if they don't exist
os.makedirs('predicted_c_code', exist_ok=True)

# Load test data
test_data = load_data('test1.src', 'test1.tgt')

# Generate predictions and calculate scores
predictions = []
references = []
semantic_similarities = []
functional_similarities = []

# Prepare CSV file for semantic similarities
with open('semantic_similarities.csv', 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(['Sr No','Verilog Source Code', 'Target C', 'Prediction C' , 'Semantic Similarity'])

    for idx, (verilog_code, target_c_code) in enumerate(tqdm(test_data, desc="Generating predictions"), 1):
        # Generate C code
        c_code = translate_verilog_to_c(verilog_code)
        predictions.append(c_code.split())
        references.append([target_c_code.split()])

        # Save predicted C code to individual file
        with open(f'predicted_c_code/prediction_{idx}.c', 'w') as f:
            f.write(c_code)

        # Calculate similarities
        semantic_sim = get_semantic_similarity(c_code, target_c_code)
        # functional_sim = get_functional_similarity(c_code, target_c_code)

        semantic_similarities.append(semantic_sim)
        # functional_similarities.append(functional_sim)

        # Write semantic similarity to CSV
        csv_writer.writerow([idx, verilog_code, target_c_code, c_code, semantic_sim])

# Calculate BLEU score
smoothie = SmoothingFunction().method4
bleu_score = corpus_bleu(references, predictions, smoothing_function=smoothie)

# Print some example translations with similarity scores
num_examples = 5
print("\nExample Translations:")
for i in range( len(test_data)):
    print(f"\nExample {i + 1}:")
    print(f"Source (Verilog):\n{test_data[i][0]}")
    print(f"Target (C):\n{test_data[i][1]}")
    print(f"Prediction (C):\n{' '.join(predictions[i])}")
    print(f"Semantic Similarity: {semantic_similarities[i]:.4f}")
    # print(f"Functional Similarity: {functional_similarities[i]:.4f}")
    print("-" * 50)

# Save predictions and scores to a file
with open('predictions_with_scores.txt', 'w') as f:
    for i, (pred, sem_sim, func_sim) in enumerate(zip(predictions, semantic_similarities, functional_similarities)):
        f.write(f"Prediction {i+1}:\n")
        f.write(' '.join(pred) + '\n')
        f.write(f"Semantic Similarity: {sem_sim:.4f}\n")
        # f.write(f"Functional Similarity: {func_sim:.4f}\n")
        f.write("-" * 50 + "\n")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/31.7k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.89G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

... (15 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating predictions:   0%|          | 0/30 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.37 GiB. GPU 0 has a total capacity of 14.75 GiB of which 1001.06 MiB is free. Process 4800 has 13.77 GiB memory in use. Of the allocated memory 13.61 GiB is allocated by PyTorch, and 34.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import csv

nltk.download('punkt')

# Initialize models and tokenizer
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize semantic similarity model
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_semantic_similarity(code1, code2):
    # Get embeddings for both code snippets
    embedding1 = semantic_model.encode(code1)
    embedding2 = semantic_model.encode(code2)

    # Calculate cosine similarity
    similarity = np.dot(embedding1, embedding2) / (np.linalg.norm(embedding1) * np.linalg.norm(embedding2))
    return similarity

def get_functional_similarity(code1, code2):
    prompt = f"""Compare the following two code snippets and rate their functional/conceptual similarity on a scale of 0 to 1, where 1 means identical functionality and 0 means completely different functionality. Only output the numerical score.

    Code 1:
    {code1}

    Code 2:
    {code2}"""

    messages = [
        {"role": "system", "content": "You are a code analysis assistant that evaluates functional similarity between code snippets."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=10,
            temperature=0.1
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    try:
        similarity_score = float(response.strip())
        return max(0.0, min(1.0, similarity_score))  # Ensure score is between 0 and 1
    except:
        return 0.0

def translate_verilog_to_c(verilog_code):
    prompt = f"""Translate the following Verilog code to C code.
    Only output the C code, without any explanations or additional text.

    Verilog code:
    {verilog_code}"""

    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant that converts Verilog to C code."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response.strip()

def load_data(src_file, tgt_file):
    with open(src_file, 'r') as sf, open(tgt_file, 'r') as tf:
        src_lines = sf.readlines()
        tgt_lines = tf.readlines()
    assert len(src_lines) == len(tgt_lines), "Mismatch in number of source and target lines"
    return [(src.strip(), tgt.strip()) for src, tgt in zip(src_lines, tgt_lines)]

def main():
    # Create output directories if they don't exist
    os.makedirs('predicted_c_code', exist_ok=True)

    # Load test data
    test_data = load_data('test1.src', 'test1.tgt')

    # Generate predictions and calculate scores
    predictions = []
    references = []
    semantic_similarities = []
    functional_similarities = []

    # Prepare CSV file for semantic similarities
    with open('semantic_similarities.csv', 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(['Sr No', 'Verilog Source Code', 'Target C', 'Prediction C', 'Semantic Similarity'])

        for idx, (verilog_code, target_c_code) in enumerate(tqdm(test_data, desc="Generating predictions"), 1):
            # Generate C code
            c_code = translate_verilog_to_c(verilog_code)
            predictions.append(c_code.split())
            references.append([target_c_code.split()])

            # Save predicted C code to individual file
            with open(f'predicted_c_code/prediction_{idx}.c', 'w') as f:
                f.write(c_code)

            # Calculate similarities
            semantic_sim = get_semantic_similarity(c_code, target_c_code)
            functional_sim = get_functional_similarity(c_code, target_c_code)

            semantic_similarities.append(semantic_sim)
            functional_similarities.append(functional_sim)

            # Write semantic similarity to CSV
            csv_writer.writerow([idx, verilog_code, target_c_code, c_code, semantic_sim])

    # Calculate BLEU score
    smoothie = SmoothingFunction().method4
    bleu_score = corpus_bleu(references, predictions, smoothing_function=smoothie)

    # Print results
    print(f"\nBLEU Score: {bleu_score:.4f}")
    print(f"Average Semantic Similarity: {np.mean(semantic_similarities):.4f}")
    print(f"Average Functional Similarity: {np.mean(functional_similarities):.4f}")

    # Print some example translations with similarity scores
    print("\nExample Translations:")
    for i in range(min(5, len(test_data))):
        print(f"\nExample {i + 1}:")
        print(f"Source (Verilog):\n{test_data[i][0]}")
        print(f"Target (C):\n{test_data[i][1]}")
        print(f"Prediction (C):\n{' '.join(predictions[i])}")
        print(f"Semantic Similarity: {semantic_similarities[i]:.4f}")
        print(f"Functional Similarity: {functional_similarities[i]:.4f}")
        print("-" * 50)

if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Generating predictions:   0%|          | 0/30 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 3.06 MiB is free. Process 4800 has 14.74 GiB memory in use. Of the allocated memory 14.52 GiB is allocated by PyTorch, and 102.33 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import csv
import gc

def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

def load_data(src_file, tgt_file):
    with open(src_file, 'r') as sf, open(tgt_file, 'r') as tf:
        src_lines = sf.readlines()
        tgt_lines = tf.readlines()
    assert len(src_lines) == len(tgt_lines), "Mismatch in number of source and target lines"
    return [(src.strip(), tgt.strip()) for src, tgt in zip(src_lines, tgt_lines)]

def initialize_models():
    clear_gpu_memory()

    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load model and move to device
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to(device)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Load semantic model and move to device
    semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
    semantic_model = semantic_model.to(device)

    return model, tokenizer, semantic_model, device

def get_semantic_similarity(semantic_model, code1, code2, device):
    try:
        with torch.no_grad():
            # Ensure inputs are strings
            code1 = str(code1) if code1 is not None else ""
            code2 = str(code2) if code2 is not None else ""

            # Get embeddings and ensure they're on the correct device
            embedding1 = semantic_model.encode(code1, convert_to_tensor=True).to(device)
            embedding2 = semantic_model.encode(code2, convert_to_tensor=True).to(device)

            # Move to CPU for numpy operations
            embedding1_cpu = embedding1.cpu().numpy()
            embedding2_cpu = embedding2.cpu().numpy()

            # Calculate similarity
            norm1 = np.linalg.norm(embedding1_cpu)
            norm2 = np.linalg.norm(embedding2_cpu)

            if norm1 == 0 or norm2 == 0:
                return 0.0

            similarity = np.dot(embedding1_cpu, embedding2_cpu) / (norm1 * norm2)
            return float(similarity)
    except Exception as e:
        print(f"Error in semantic similarity calculation: {str(e)}")
        return 0.0

def translate_verilog_to_c(model, tokenizer, device, verilog_code):
    try:
        prompt = f"""Translate the following Verilog code to C code.
        Only output the C code, without any explanations or additional text.

        Verilog code:
        {verilog_code}"""

        messages = [
            {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant that converts Verilog to C code."},
            {"role": "user", "content": prompt}
        ]

        # Create input text
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize and move to device
        inputs = tokenizer([text], return_tensors="pt")
        input_ids = inputs.input_ids.to(device)
        attention_mask = inputs.attention_mask.to(device) if 'attention_mask' in inputs else None

        with torch.no_grad():
            # Generate
            generate_kwargs = {
                'max_new_tokens': 1024,
                'temperature': 0.7,
                'do_sample': True
            }
            if attention_mask is not None:
                generate_kwargs['attention_mask'] = attention_mask

            generated_ids = model.generate(
                input_ids,
                **generate_kwargs
            )

            # Process output
            generated_ids = generated_ids[:, len(input_ids[0]):]
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

            # Clear memory
            del inputs, input_ids, attention_mask, generated_ids
            clear_gpu_memory()

            return response.strip()

    except Exception as e:
        print(f"Error in translation: {str(e)}")
        print(f"Current device - Model: {next(model.parameters()).device}")
        return ""

def process_batch(model, tokenizer, semantic_model, device, batch_data, start_idx, csv_writer):
    predictions = []
    references = []
    semantic_similarities = []

    for idx, (verilog_code, target_c_code) in enumerate(batch_data, start_idx):
        try:
            # Print device information for debugging
            print(f"\nProcessing item {idx}")
            print(f"Model device: {next(model.parameters()).device}")
            print(f"Semantic model device: {next(semantic_model.parameters()).device}")

            # Generate C code
            c_code = translate_verilog_to_c(model, tokenizer, device, verilog_code)

            if not c_code:  # Skip if translation failed
                print(f"Skipping item {idx} due to empty translation")
                continue

            predictions.append(c_code.split())
            references.append([target_c_code.split()])

            # Save predicted C code
            with open(f'predicted_c_code/prediction_{idx}.c', 'w') as f:
                f.write(c_code)

            # Calculate semantic similarity
            semantic_sim = get_semantic_similarity(semantic_model, c_code, target_c_code, device)
            semantic_similarities.append(semantic_sim)

            # Write to CSV
            csv_writer.writerow([idx, verilog_code, target_c_code, c_code, semantic_sim])

        except Exception as e:
            print(f"Error processing item {idx}: {str(e)}")
            continue

        # Clear memory periodically
        if idx % 2 == 0:
            clear_gpu_memory()

    return predictions, references, semantic_similarities

def main():
    try:
        # Create output directory
        os.makedirs('predicted_c_code', exist_ok=True)

        # Initialize models
        model, tokenizer, semantic_model, device = initialize_models()
        print(f"Initial model device: {next(model.parameters()).device}")
        print(f"Initial semantic model device: {next(semantic_model.parameters()).device}")

        # Load test data
        test_data = load_data('test1.src', 'test1.tgt')

        # Process in smaller batches
        batch_size = 3  # Reduced batch size
        all_predictions = []
        all_references = []
        all_semantic_similarities = []

        with open('semantic_similarities.csv', 'w', newline='') as csvfile:
            csv_writer = csv.writer(csvfile)
            csv_writer.writerow(['Sr No', 'Verilog Source Code', 'Target C', 'Prediction C', 'Semantic Similarity'])

            for batch_start in tqdm(range(0, len(test_data), batch_size), desc="Processing batches"):
                batch_end = min(batch_start + batch_size, len(test_data))
                batch = test_data[batch_start:batch_end]

                # Ensure models are on correct device before each batch
                model = model.to(device)
                semantic_model = semantic_model.to(device)

                predictions, references, similarities = process_batch(
                    model, tokenizer, semantic_model, device,
                    batch, batch_start + 1, csv_writer
                )

                all_predictions.extend(predictions)
                all_references.extend(references)
                all_semantic_similarities.extend(similarities)

                # Clear memory after each batch
                clear_gpu_memory()

        # Calculate final scores only if we have results
        if all_predictions and all_references:
            smoothie = SmoothingFunction().method4
            bleu_score = corpus_bleu(all_references, all_predictions, smoothing_function=smoothie)
            avg_semantic_similarity = np.mean(all_semantic_similarities) if all_semantic_similarities else 0.0

            print(f"\nProcessing completed:")
            print(f"Total samples processed: {len(all_predictions)}")
            print(f"BLEU Score: {bleu_score:.4f}")
            print(f"Average Semantic Similarity: {avg_semantic_similarity:.4f}")

            # Print example translations
            print("\nExample Translations:")
            for i in range(min(5, len(all_predictions))):
                print(f"\nExample {i + 1}:")
                print(f"Source (Verilog):\n{test_data[i][0]}")
                print(f"Target (C):\n{test_data[i][1]}")
                print(f"Prediction (C):\n{' '.join(all_predictions[i])}")
                print(f"Semantic Similarity: {all_semantic_similarities[i]:.4f}")
                print("-" * 50)
        else:
            print("No successful translations to evaluate.")

    except Exception as e:
        print(f"Error in main execution: {str(e)}")

if __name__ == "__main__":
    clear_gpu_memory()
    main()

Using device: cuda
Initial model device: cuda:0
Initial semantic model device: cuda:0


Processing batches:   0%|          | 0/10 [00:00<?, ?it/s]


Processing item 1
Model device: cuda:0
Semantic model device: cuda:0

Processing item 2
Model device: cuda:0
Semantic model device: cuda:0

Processing item 3
Model device: cuda:0
Semantic model device: cuda:0


Processing batches:  10%|█         | 1/10 [01:08<10:14, 68.33s/it]


Processing item 4
Model device: cuda:0
Semantic model device: cuda:0

Processing item 5
Model device: cuda:0
Semantic model device: cuda:0

Processing item 6
Model device: cuda:0
Semantic model device: cuda:0


Processing batches:  20%|██        | 2/10 [02:03<08:03, 60.47s/it]

... (17 outputs left out)

Processing batches:  90%|█████████ | 9/10 [12:36<01:18, 78.45s/it]


Processing item 28
Model device: cuda:0
Semantic model device: cuda:0

Processing item 29
Model device: cuda:0
Semantic model device: cuda:0

Processing item 30
Model device: cuda:0
Semantic model device: cuda:0


Processing batches: 100%|██████████| 10/10 [15:00<00:00, 90.07s/it]


Processing completed:
Total samples processed: 30
BLEU Score: 0.0030
Average Semantic Similarity: 0.4550

Example Translations:

Example 1:
Source (Verilog):
module entry_point ( , ap_clk, , ap_rst, , ap_start, , ap_done, , ap_idle, , ap_ready, , n, , item, , temptemp1, , temptemp1_ap_vld, , temptemp2, , temptemp2_ap_vld, , temptemp3, , temptemp3_ap_vld, , ap_return , ); , parameter ap_ST_fsm_state1 = 3'd1; , parameter ap_ST_fsm_state2 = 3'd2; , parameter ap_ST_fsm_state3 = 3'd4; , input ap_clk; , input ap_rst; , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] n; , input [31:0] item; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] temptemp2; , output temptemp2_ap_vld; , output [31:0] temptemp3; , output temptemp3_ap_vld; , output [31:0] ap_return; , reg ap_done; , reg ap_idle; , reg ap_ready; , reg temptemp1_ap_vld; , reg temptemp2_ap_vld; , reg temptemp3_ap_vld; , (* fsm_encoding = "none" *) reg [2:0] ap_CS_fsm; , wire ap_CS